## EE 460 Final Project

### Members:
Aaron Hui  
Osher Nodel  
Theodore Matsusaka  
Nathan Spare

##### Imports

In [61]:
import numpy as np
import autograd.numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import seaborn as sns
import copy
from sklearn.model_selection import train_test_split
readDataPath = './readData/'
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from scipy.fft import rfft, rfftfreq
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
import matplotlib
from sklearn.neural_network import MLPRegressor
from autograd import grad
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings
from sklearn.exceptions import ConvergenceWarning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import optuna
from sklearn.model_selection import cross_val_score
import os

##### Datasets and Feature Extraction

In [39]:
def extract_vibration_features(signal, sample_rate):
    """
    Extracts time and frequency features from a 1-second 1D vibration signal.
    """
    # 1. Time-Domain Features
    rms = np.sqrt(np.mean(signal**2))
    p2p = np.ptp(signal) # Peak-to-Peak
    skewness = skew(signal)
    kurt = kurtosis(signal)
    
    # 2. Frequency-Domain Features (Fast Fourier Transform)
    # rfft is highly efficient for real-valued inputs
    fft_amplitudes = np.abs(rfft(signal))
    
    # Get the top 3 dominant frequency amplitudes (excluding the 0Hz DC component)
    fft_amplitudes[0] = 0 
    top_indices = np.argsort(fft_amplitudes)[-3:]
    top_amps = fft_amplitudes[top_indices]
    
    return {
        'Vib_RMS': rms,
        'Vib_P2P': p2p,
        'Vib_Skewness': skewness,
        'Vib_Kurtosis': kurt,
        'FFT_Amp_1': top_amps[2], # Largest peak
        'FFT_Amp_2': top_amps[1],
        'FFT_Amp_3': top_amps[0]
    }

In [40]:
def process_kaggle_file(filepath, sample_rate=25600):
    # 1. Tell Pandas there is no header row
    df = pd.read_csv(filepath, header=None)
    
    # 2. Manually name the columns based on the dataset manual
    df.columns = ['Vibration_X', 'Vibration_Y', 'Bearing_Temp', 'Atmospheric_Temp']
    
    # Calculate how many full 1-second windows we have
    total_samples = len(df)
    num_windows = total_samples // sample_rate
    
    features_list = []
    
    for i in range(num_windows):
        start_idx = i * sample_rate
        end_idx = start_idx + sample_rate
        window = df.iloc[start_idx:end_idx]
        
        # Now these keys will work perfectly!
        vib_signal = window['Vibration_X'].values
        vib_features = extract_vibration_features(vib_signal, sample_rate)
        
        avg_bearing_temp = window['Bearing_Temp'].mean()
        avg_ambient_temp = window['Atmospheric_Temp'].mean()
        delta_t = avg_bearing_temp - avg_ambient_temp
        
        vib_features['Delta_T'] = delta_t
        vib_features['Window_ID'] = i 
        
        features_list.append(vib_features)
        
    return pd.DataFrame(features_list)

In [41]:
def process_entire_kaggle_directory(directory_path, sample_rate=25600):
    """
    Iterates through all CSVs in a folder, extracts features, 
    and combines the results into one lightweight DataFrame.
    """
    # Find all CSV files in the target directory
    # sorted() ensures we process them in chronological order
    file_pattern = os.path.join(directory_path, "*.csv")
    filepaths = sorted(glob.glob(file_pattern))
    
    all_features = []
    
    for filepath in filepaths:
        print(f"Processing: {os.path.basename(filepath)}...")
        
        # Call the processing function we wrote earlier
        # This returns a DataFrame of extracted features for this specific file
        file_features_df = process_kaggle_file(filepath, sample_rate)
        
        all_features.append(file_features_df)
        
    # Concatenate all the small feature DataFrames into one master DataFrame
    master_feature_df = pd.concat(all_features, ignore_index=True)
    
    return master_feature_df

# Example Execution:
# final_kaggle_dataset = process_entire_kaggle_directory('./kaggle_raw_data/')
# final_kaggle_dataset.to_csv('cleaned_kaggle_features.csv', index=False)

In [42]:
def process_nasa_directory(directory_path, target_column, test_id, sample_rate=20480):
    """
    Processes a single NASA test directory, targeting the specific column
    that contains the failing bearing.
    """
    features_list = []
    filenames = sorted(os.listdir(directory_path))
    
    for i, filename in enumerate(filenames):
        filepath = os.path.join(directory_path, filename)
        
        try:
            # NASA data is tab-separated
            df = pd.read_csv(filepath, sep='\t', header=None)
            
            # Extract the specific column for the failing bearing
            vib_signal = df[target_column].values 
            
            # Use the shared feature extraction function we defined earlier
            vib_features = extract_vibration_features(vib_signal, sample_rate)
            
            # Track which file and which test this came from
            vib_features['File_ID'] = i 
            vib_features['Test_ID'] = test_id 
            
            features_list.append(vib_features)
            
        except Exception as e:
            # Silently skip non-data files or read errors
            continue
            
    return pd.DataFrame(features_list)

In [43]:
def compile_all_nasa_tests(base_nasa_path):
    """
    Processes all 3 NASA tests and concatenates them into one master dataset.
    """
    # ---------------------------------------------------------
    # Mapping the Failing Bearings (0-indexed columns)
    # ---------------------------------------------------------
    # Test 1: 8 columns (2 per bearing). Bearing 3 failed. 
    # Columns 4 and 5 represent Bearing 3. We will use Column 4.
    test1_path = os.path.join(base_nasa_path, '1st_test')
    df_test1 = process_nasa_directory(test1_path, target_column=4, test_id=1)
    print(f"Test 1 processed: {len(df_test1)} files.")

    # Test 2: 4 columns (1 per bearing). Bearing 1 failed.
    # Column 0 represents Bearing 1.
    test2_path = os.path.join(base_nasa_path, '2nd_test')
    df_test2 = process_nasa_directory(test2_path, target_column=0, test_id=2)
    print(f"Test 2 processed: {len(df_test2)} files.")

    # Test 3: 4 columns (1 per bearing). Bearing 3 failed.
    # Column 2 represents Bearing 3.
    test3_path = os.path.join(base_nasa_path, '3rd_test')
    df_test3 = process_nasa_directory(test3_path, target_column=2, test_id=3)
    print(f"Test 3 processed: {len(df_test3)} files.")

    # ---------------------------------------------------------
    # Combine everything into one massive Vibration DataFrame
    # ---------------------------------------------------------
    nasa_master_df = pd.concat([df_test1, df_test2, df_test3], ignore_index=True)
    
    return nasa_master_df

# Example execution:
# nasa_combined_features = compile_all_nasa_tests('./nasa_ims_dataset/')

In [44]:
def normalize_datasets(kaggle_df, nasa_df):
    # Separate the Temp data from Kaggle (so it isn't fed to the Vibration Model)
    kaggle_temp_features = kaggle_df[['Delta_T']]
    kaggle_vib_features = kaggle_df.drop(columns=['Delta_T', 'Window_ID'])
    
    nasa_vib_features = nasa_df.drop(columns=['File_ID'])
    
    # 1. Initialize Independent Scalers
    kaggle_vib_scaler = StandardScaler()
    nasa_vib_scaler = StandardScaler()
    temp_scaler = StandardScaler()
    
    # 2. Fit and Transform Independently
    # This transforms the absolute physics of each rig into a relative Z-score (Mean=0, SD=1)
    kaggle_vib_normalized = pd.DataFrame(
        kaggle_vib_scaler.fit_transform(kaggle_vib_features), 
        columns=kaggle_vib_features.columns
    )
    
    nasa_vib_normalized = pd.DataFrame(
        nasa_vib_scaler.fit_transform(nasa_vib_features), 
        columns=nasa_vib_features.columns
    )
    
    kaggle_temp_normalized = pd.DataFrame(
        temp_scaler.fit_transform(kaggle_temp_features),
        columns=kaggle_temp_features.columns
    )
    
    return kaggle_vib_normalized, nasa_vib_normalized, kaggle_temp_normalized

# Execute normalization
# kaggle_vib_norm, nasa_vib_norm, kaggle_temp_norm = normalize_datasets(kaggle_features_df, nasa_features_df)

In [ ]:
# final_kaggle_dataset = process_entire_kaggle_directory('./kaggle_raw_data/')
# final_kaggle_dataset.to_csv('cleaned_kaggle_features.csv', index=False)
# nasa_combined_features = compile_all_nasa_tests('./nasa_ims_dataset/')
# nasa_combined_features.to_csv('compiled_nasa_features.csv', index=False)
final_kaggle_dataset = pd.read_csv('cleaned_kaggle_features.csv')
nasa_combined_features = pd.read_csv('compiled_nasa_features.csv')
kaggle_vib_norm, nasa_vib_norm, kaggle_temp_norm = normalize_datasets(final_kaggle_dataset, nasa_combined_features)

Test 1 processed: 2156 files.
Test 2 processed: 984 files.
Test 3 processed: 0 files.


##### Logistic Regression

In [46]:
def append_labels(df, anomaly_threshold=0.9):
    """
    Assumes the DataFrame is sorted chronologically.
    Marks the final 10% of the dataset as 'Anomaly' (1) and the rest as 'Healthy' (0).
    """
    total_samples = len(df)
    degradation_start_idx = int(total_samples * anomaly_threshold)
    
    # Create an array of 0s, then flip the final portion to 1s
    labels = np.zeros(total_samples)
    labels[degradation_start_idx:] = 1
    
    # Add to dataframe
    df_labeled = df.copy()
    df_labeled['Target'] = labels
    return df_labeled

In [47]:
nasa_labeled = append_labels(nasa_vib_norm, anomaly_threshold=0.9)
kaggle_labeled = append_labels(kaggle_temp_norm, anomaly_threshold=0.9)

# Split NASA (Vibration Expert)
X_vib = nasa_labeled.drop(columns=['Target'])
y_vib = nasa_labeled['Target']
X_vib_train, X_vib_test, y_vib_train, y_vib_test = train_test_split(
    X_vib, y_vib, test_size=0.2, random_state=42, shuffle=True
)

# Split Kaggle (Thermodynamics Expert)
X_temp = kaggle_labeled.drop(columns=['Target'])
y_temp = kaggle_labeled['Target']
X_temp_train, X_temp_test, y_temp_train, y_temp_test = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, shuffle=True
)

In [48]:
print("Training Vibration Expert...")
vib_model = LogisticRegression(max_iter=1000)
vib_model.fit(X_vib_train, y_vib_train)
print(classification_report(y_vib_test, vib_model.predict(X_vib_test)))

print("\nTraining Thermodynamics Expert...")
temp_model = LogisticRegression(max_iter=1000)
temp_model.fit(X_temp_train, y_temp_train)
print(classification_report(y_temp_test, temp_model.predict(X_temp_test)))

Training Vibration Expert...
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98       565
         1.0       0.91      0.78      0.84        63

    accuracy                           0.97       628
   macro avg       0.94      0.88      0.91       628
weighted avg       0.97      0.97      0.97       628


Training Thermodynamics Expert...
              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98      1817
         1.0       0.95      0.73      0.83       196

    accuracy                           0.97      2013
   macro avg       0.96      0.86      0.91      2013
weighted avg       0.97      0.97      0.97      2013



In [49]:
def get_agent_telemetry(vib_sample, temp_sample):
    """
    Takes a single row of new vibration features and new temperature features,
    and returns the risk probabilities for the AI Decision Agent.
    """
    # .predict_proba() returns an array: [Probability of 0, Probability of 1]
    # We only want the probability of 1 (Anomaly Risk)
    vib_risk = vib_model.predict_proba(vib_sample)[0][1]
    temp_risk = temp_model.predict_proba(temp_sample)[0][1]
    
    return {
        "Vibration_Anomaly_Confidence": round(vib_risk, 4),
        "Thermal_Anomaly_Confidence": round(temp_risk, 4)
    }

# --- Example Usage for the AI Agent ---
# Grabbing a random single sample from the test sets to simulate real-time data
sample_vib = X_vib_test.iloc[[0]] 
sample_temp = X_temp_test.iloc[[0]]

agent_data = get_agent_telemetry(sample_vib, sample_temp)
print(f"\n--- AI Agent Live Feed ---\n{agent_data}")


--- AI Agent Live Feed ---
{'Vibration_Anomaly_Confidence': np.float64(0.9622), 'Thermal_Anomaly_Confidence': np.float64(0.0025)}


##### MLP

In [50]:
# Ignore convergence warnings for a cleaner terminal output
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ---------------------------------------------------------
# 1. Train the Vibration Expert (Deep MLP)
# ---------------------------------------------------------
# We have 7 extracted vibration features. A "funnel" architecture 
# (e.g., 16 nodes in the first hidden layer, 8 in the second) works well here.
print("Training Vibration Expert (MLP)...")

vib_mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',      # Adaptive moment estimation (stochastic gradient descent)
    alpha=0.0001,       # L2 penalty (regularization term) to prevent overfitting
    batch_size=64,
    max_iter=500,
    random_state=42
)

vib_mlp.fit(X_vib_train, y_vib_train)
print(classification_report(y_vib_test, vib_mlp.predict(X_vib_test)))

Training Vibration Expert (MLP)...
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       565
         1.0       0.95      0.94      0.94        63

    accuracy                           0.99       628
   macro avg       0.97      0.97      0.97       628
weighted avg       0.99      0.99      0.99       628



In [51]:
print("\nTraining Thermodynamics Expert (MLP)...")

temp_mlp = MLPClassifier(
    hidden_layer_sizes=(4, 2),
    activation='relu',
    solver='lbfgs',     # An optimizer in the family of quasi-Newton methods. 
                        # It often converges faster and performs better on very small datasets or 1D feature spaces.
    max_iter=500,
    random_state=42
)

temp_mlp.fit(X_temp_train, y_temp_train)
print(classification_report(y_temp_test, temp_mlp.predict(X_temp_test)))


Training Thermodynamics Expert (MLP)...
              precision    recall  f1-score   support

         0.0       0.90      1.00      0.95      1817
         1.0       0.00      0.00      0.00       196

    accuracy                           0.90      2013
   macro avg       0.45      0.50      0.47      2013
weighted avg       0.81      0.90      0.86      2013



/home/aaronhui/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/aaronhui/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/aaronhui/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [52]:
def get_agent_telemetry(vib_sample, temp_sample):
    """
    Because MLPs in scikit-learn also support predict_proba, 
    the AI agent's telemetry function remains exactly the same!
    """
    vib_risk = vib_mlp.predict_proba(vib_sample)[0][1]
    temp_risk = temp_mlp.predict_proba(temp_sample)[0][1]
    
    return {
        "Vibration_Anomaly_Confidence": round(vib_risk, 4),
        "Thermal_Anomaly_Confidence": round(temp_risk, 4)
    }

# --- Example Execution ---
sample_vib = X_vib_test.iloc[[0]] 
sample_temp = X_temp_test.iloc[[0]]

agent_data = get_agent_telemetry(sample_vib, sample_temp)
print(f"\n--- AI Agent Live Feed ---\n{agent_data}")


--- AI Agent Live Feed ---
{'Vibration_Anomaly_Confidence': np.float64(0.9997), 'Thermal_Anomaly_Confidence': np.float64(0.1008)}


##### LSTM Autoencoder

In [53]:
def create_sequences(X, y, time_steps=5):
    """
    Converts 2D tabular data into 3D sequential data for LSTM input.
    time_steps: How many previous windows the LSTM should look at to make a decision.
    """
    Xs, ys = [], []
    # Slide a window of size 'time_steps' across the dataset
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        # The label is the target variable immediately following the sequence
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)

# Define our memory window (e.g., looking at 5 consecutive segments)
TIME_STEPS = 5

# Create 3D sequences for both models
X_vib_seq, y_vib_seq = create_sequences(X_vib, y_vib, TIME_STEPS)
X_temp_seq, y_temp_seq = create_sequences(X_temp, y_temp, TIME_STEPS)

# Standard Train/Test Split (Note: For strict time-series forecasting, 
# you'd usually split chronologically, but random split is okay for anomaly detection here)
from sklearn.model_selection import train_test_split

X_vib_train_seq, X_vib_test_seq, y_vib_train_seq, y_vib_test_seq = train_test_split(
    X_vib_seq, y_vib_seq, test_size=0.2, random_state=42
)

X_temp_train_seq, X_temp_test_seq, y_temp_train_seq, y_temp_test_seq = train_test_split(
    X_temp_seq, y_temp_seq, test_size=0.2, random_state=42
)

In [54]:
def create_dataloaders(X_train, y_train, X_test, y_test, batch_size=64):
    # Convert numpy arrays to PyTorch tensors (Float32 is standard for NN weights)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # Add dimension for binary output
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # Wrap tensors into Datasets
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    # Create DataLoaders for efficient batch iteration
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

# Create the loaders
vib_train_loader, vib_test_loader = create_dataloaders(
    X_vib_train_seq, y_vib_train_seq, X_vib_test_seq, y_vib_test_seq
)

temp_train_loader, temp_test_loader = create_dataloaders(
    X_temp_train_seq, y_temp_train_seq, X_temp_test_seq, y_temp_test_seq
)

In [55]:
class BearingLSTMExpert(nn.Module):
    def __init__(self, input_features, hidden_size=64, num_layers=1, dense_units=16):
        super(BearingLSTMExpert, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Define the LSTM layer (batch_first=True makes inputs [batch, seq, feature])
        self.lstm = nn.LSTM(
            input_size=input_features, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True
        )
        
        self.dropout = nn.Dropout(0.2)
        
        # Fully connected layers to interpret the LSTM output
        self.fc1 = nn.Linear(hidden_size, dense_units)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(dense_units, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Initialize hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))
        
        # We only want the output from the final time step in the sequence
        out = out[:, -1, :]
        
        # Pass through dense layers
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        
        return out

# Initialize the Models
vib_features = X_vib_train_seq.shape[2]
temp_features = X_temp_train_seq.shape[2]

vib_lstm_pt = BearingLSTMExpert(input_features=vib_features, hidden_size=64)
temp_lstm_pt = BearingLSTMExpert(input_features=temp_features, hidden_size=16, dense_units=8)

In [56]:
import torch.optim as optim

def train_pytorch_model(model, train_loader, epochs=100, lr=0.001):
    # Standard binary cross entropy loss for 0/1 targets
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Set model to training mode
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            # 1. Clear previous gradients
            optimizer.zero_grad()
            
            # 2. Forward pass (predict)
            predictions = model(batch_X)
            
            # 3. Calculate loss
            loss = criterion(predictions, batch_y)
            
            # 4. Backward pass (calculate gradients)
            loss.backward()
            
            # 5. Update weights
            optimizer.step()
            
            epoch_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")

print("\n--- Training PyTorch Vibration Expert ---")
train_pytorch_model(vib_lstm_pt, vib_train_loader, epochs=20)

print("\n--- Training PyTorch Thermodynamics Expert ---")
train_pytorch_model(temp_lstm_pt, temp_train_loader, epochs=20)


--- Training PyTorch Vibration Expert ---
Epoch 1/20 | Loss: 0.5842
Epoch 2/20 | Loss: 0.1108
Epoch 3/20 | Loss: 0.0254
Epoch 4/20 | Loss: 0.0236
Epoch 5/20 | Loss: 0.0247
Epoch 6/20 | Loss: 0.0213
Epoch 7/20 | Loss: 0.0205
Epoch 8/20 | Loss: 0.0180
Epoch 9/20 | Loss: 0.0166
Epoch 10/20 | Loss: 0.0168
Epoch 11/20 | Loss: 0.0159
Epoch 12/20 | Loss: 0.0162
Epoch 13/20 | Loss: 0.0149
Epoch 14/20 | Loss: 0.0170
Epoch 15/20 | Loss: 0.0147
Epoch 16/20 | Loss: 0.0142
Epoch 17/20 | Loss: 0.0136
Epoch 18/20 | Loss: 0.0133
Epoch 19/20 | Loss: 0.0122
Epoch 20/20 | Loss: 0.0124

--- Training PyTorch Thermodynamics Expert ---
Epoch 1/20 | Loss: 0.5234
Epoch 2/20 | Loss: 0.1360
Epoch 3/20 | Loss: 0.0974
Epoch 4/20 | Loss: 0.0935
Epoch 5/20 | Loss: 0.0915
Epoch 6/20 | Loss: 0.0897
Epoch 7/20 | Loss: 0.0896
Epoch 8/20 | Loss: 0.0889
Epoch 9/20 | Loss: 0.0869
Epoch 10/20 | Loss: 0.0875
Epoch 11/20 | Loss: 0.0868
Epoch 12/20 | Loss: 0.0868
Epoch 13/20 | Loss: 0.0874
Epoch 14/20 | Loss: 0.0864
Epoch 15/

In [57]:
def get_pytorch_agent_telemetry(vib_sequence, temp_sequence):
    """
    Evaluates new 3D sequences using PyTorch and returns the anomaly risks.
    """
    # Convert numpy sequences to PyTorch tensors
    vib_tensor = torch.tensor(vib_sequence, dtype=torch.float32)
    temp_tensor = torch.tensor(temp_sequence, dtype=torch.float32)
    
    # Set models to evaluation mode (turns off dropout, etc.)
    vib_lstm_pt.eval()
    temp_lstm_pt.eval()
    
    # Disable gradient calculation for faster inference
    with torch.no_grad():
        vib_risk = vib_lstm_pt(vib_tensor).item()
        temp_risk = temp_lstm_pt(temp_tensor).item()
        
    return {
        "PyTorch_Vibration_Anomaly_Confidence": round(vib_risk, 4),
        "PyTorch_Thermal_Anomaly_Confidence": round(temp_risk, 4)
    }

# --- Example Execution ---
sample_vib_seq = X_vib_test_seq[0:1] 
sample_temp_seq = X_temp_test_seq[0:1]

agent_data = get_pytorch_agent_telemetry(sample_vib_seq, sample_temp_seq)
print(f"\n--- AI Agent PyTorch Telemetry Feed ---\n{agent_data}")


--- AI Agent PyTorch Telemetry Feed ---
{'PyTorch_Vibration_Anomaly_Confidence': 0.0001, 'PyTorch_Thermal_Anomaly_Confidence': 0.0605}


##### Hyperparameter Optimization and Optuna

In [58]:
def objective_lr(trial):
    # Let Optuna choose the regularization strength (log scale)
    c_value = trial.suggest_float("C", 1e-4, 1e2, log=True)
    # Let Optuna pick the solver algorithm
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])
    
    model = LogisticRegression(C=c_value, solver=solver, max_iter=1000)
    
    # Cross-validation returns an array of scores; we return the mean for Optuna to evaluate
    score = cross_val_score(model, X_vib_train, y_vib_train, n_jobs=-1, cv=3, scoring="accuracy").mean()
    return score

In [59]:
def objective_mlp(trial):
    # Optuna decides how many hidden layers (1 to 3)
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []
    
    # Optuna decides the node count for EACH layer dynamically
    for i in range(n_layers):
        layers.append(trial.suggest_int(f"n_units_l{i}", 8, 64))
        
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    learning_rate = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)
    
    model = MLPClassifier(
        hidden_layer_sizes=tuple(layers), 
        alpha=alpha, 
        learning_rate_init=learning_rate, 
        max_iter=500
    )
    
    score = cross_val_score(model, X_vib_train, y_vib_train, n_jobs=-1, cv=3, scoring="accuracy").mean()
    return score

In [60]:
print("Optimizing Logistic Regression...")
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_lr, n_trials=20)
print("Best LR Params:", study_lr.best_params)

print("\nOptimizing MLP...")
study_mlp = optuna.create_study(direction="maximize")
study_mlp.optimize(objective_mlp, n_trials=20)
print("Best MLP Params:", study_mlp.best_params)

[I 2026-05-06 13:15:08,121] A new study created in memory with name: no-name-35316b76-90b1-4bdb-a0e7-e9997661619b


Optimizing Logistic Regression...


[I 2026-05-06 13:15:11,661] Trial 0 finished with value: 0.98407912107966 and parameters: {'C': 1.3695960720557072, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.98407912107966.
[I 2026-05-06 13:15:15,086] Trial 1 finished with value: 0.9852729137380253 and parameters: {'C': 4.135449203868084, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.9852729137380253.
[I 2026-05-06 13:15:17,641] Trial 2 finished with value: 0.9796983962688276 and parameters: {'C': 0.15891389841777634, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.9852729137380253.
[I 2026-05-06 13:15:17,672] Trial 3 finished with value: 0.9609855062545801 and parameters: {'C': 0.007213894468011858, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.9852729137380253.
[I 2026-05-06 13:15:17,763] Trial 4 finished with value: 0.9793001485587519 and parameters: {'C': 0.12124949182259495, 'solver': 'saga'}. Best is trial 1 with value: 0.9852729137380253.
[I 2026-05-06 13:15:17,789] Trial 5 finished with value: 0.9852729137380

Best LR Params: {'C': 93.83312081981863, 'solver': 'liblinear'}

Optimizing MLP...


[I 2026-05-06 13:15:19,789] Trial 0 finished with value: 0.9892520641872657 and parameters: {'n_layers': 2, 'n_units_l0': 44, 'n_units_l1': 39, 'alpha': 2.104524847834681e-05, 'learning_rate_init': 0.00011839652306990257}. Best is trial 0 with value: 0.9892520641872657.
[I 2026-05-06 13:15:20,304] Trial 1 finished with value: 0.9900480843714862 and parameters: {'n_layers': 3, 'n_units_l0': 34, 'n_units_l1': 23, 'n_units_l2': 27, 'alpha': 0.01827172370398911, 'learning_rate_init': 0.007231420975880859}. Best is trial 1 with value: 0.9900480843714862.
[I 2026-05-06 13:15:20,768] Trial 2 finished with value: 0.9900485596074171 and parameters: {'n_layers': 1, 'n_units_l0': 25, 'alpha': 0.00010711092414393761, 'learning_rate_init': 0.0038209743924790176}. Best is trial 2 with value: 0.9900485596074171.
[I 2026-05-06 13:15:22,300] Trial 3 finished with value: 0.9888547669490518 and parameters: {'n_layers': 3, 'n_units_l0': 31, 'n_units_l1': 56, 'n_units_l2': 54, 'alpha': 0.001483387262757313

Best MLP Params: {'n_layers': 3, 'n_units_l0': 12, 'n_units_l1': 28, 'n_units_l2': 38, 'alpha': 0.0016957577786984386, 'learning_rate_init': 0.004303783690983964}


In [62]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim):
        super(LSTMAutoencoder, self).__init__()
        # ENCODER
        self.encoder = nn.LSTM(
            input_size=n_features, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        # DECODER
        self.decoder = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=n_features, 
            batch_first=True
        )

    def forward(self, x):
        # Compress
        _, (hidden, _) = self.encoder(x)
        
        # Repeat the hidden state for the length of the sequence so the decoder can rebuild it
        hidden = hidden[-1].unsqueeze(1).repeat(1, x.shape[1], 1)
        
        # Reconstruct
        reconstructed, _ = self.decoder(hidden)
        return reconstructed

In [63]:
# Assuming X_vib_train_seq contains ONLY early, healthy sequential data
# Split it into train and validation for the Autoencoder
limit = int(len(X_vib_train_seq) * 0.8)
healthy_train = torch.tensor(X_vib_train_seq[:limit], dtype=torch.float32)
healthy_val = torch.tensor(X_vib_train_seq[limit:], dtype=torch.float32)

def objective_autoencoder(trial):
    # Hyperparameters to tune
    hidden_dim = trial.suggest_int("hidden_dim", 16, 128, step=16)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    
    # Initialize Model
    seq_len = healthy_train.shape[1]
    n_features = healthy_train.shape[2]
    model = LSTMAutoencoder(seq_len, n_features, hidden_dim)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Create simple batch loader
    train_loader = torch.utils.data.DataLoader(healthy_train, batch_size=batch_size, shuffle=True)
    
    # Train for a limited number of epochs for the trial
    epochs = 10 
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch) # Target is the input itself
            loss.backward()
            optimizer.step()
            
    # Evaluate on Validation Set
    model.eval()
    with torch.no_grad():
        val_reconstructed = model(healthy_val)
        val_loss = criterion(val_reconstructed, healthy_val).item()
        
    return val_loss

print("\nOptimizing LSTM Autoencoder...")
study_ae = optuna.create_study(direction="minimize")
# In PyTorch, trials take longer, so we limit n_trials or use timeout
study_ae.optimize(objective_autoencoder, n_trials=15) 

print("Best Autoencoder Params:", study_ae.best_params)

[I 2026-05-06 13:22:36,661] A new study created in memory with name: no-name-1a24c656-9110-4fd3-9368-31ee80869fc8



Optimizing LSTM Autoencoder...


[I 2026-05-06 13:22:37,965] Trial 0 finished with value: 0.35685741901397705 and parameters: {'hidden_dim': 112, 'lr': 0.0002059044570065428, 'batch_size': 64}. Best is trial 0 with value: 0.35685741901397705.
[I 2026-05-06 13:22:39,317] Trial 1 finished with value: 0.3379189968109131 and parameters: {'hidden_dim': 112, 'lr': 0.0016778257392452206, 'batch_size': 64}. Best is trial 1 with value: 0.3379189968109131.
[I 2026-05-06 13:22:40,473] Trial 2 finished with value: 0.3385951519012451 and parameters: {'hidden_dim': 112, 'lr': 0.001311841886610481, 'batch_size': 64}. Best is trial 1 with value: 0.3379189968109131.
[I 2026-05-06 13:22:41,888] Trial 3 finished with value: 0.33518967032432556 and parameters: {'hidden_dim': 80, 'lr': 0.007404477929519895, 'batch_size': 32}. Best is trial 3 with value: 0.33518967032432556.
[I 2026-05-06 13:22:42,279] Trial 4 finished with value: 0.4146216809749603 and parameters: {'hidden_dim': 16, 'lr': 0.00122568695377385, 'batch_size': 128}. Best is t

Best Autoencoder Params: {'hidden_dim': 80, 'lr': 0.007404477929519895, 'batch_size': 32}
